# core

> what kind of thing a document is, decided by cues before a model is asked

In [ ]:
#| default_exp core

Twenty-odd doctypes, scored from countable evidence: money, dates, reference numbers, the
cue phrases each kind uses, and the entity labels it expects. A model is asked only when two
kinds tie.

In [ ]:
#| export
import re
from collections import Counter
from fastcore.all import AttrDict, L
from rahasya import NER_CHARS, PERSON_HON

## Signals: what a document contains

In [ ]:
#| export
SIGNALS = dict(
    # what a document *contains*, as cheaply as a regex can tell: the evidence a type is scored on
    money    = r'[$€£¥₹]\s?\d[\d,]*(?:\.\d+)?|\b\d[\d,]*\.\d{2}\s?(?:usd|eur|gbp|inr|jpy|cad|aud)\b'
               r'|\b(?:usd|eur|gbp|inr|jpy|cad|aud)\s?\d[\d,]*',
    date     = r'\b\d{4}-\d{2}-\d{2}\b|\b\d{1,2}[/.]\d{1,2}[/.]\d{2,4}\b'
               r'|\b\d{1,2} (?:jan|feb|mar|apr|may|jun|jul|aug|sep|oct|nov|dec)[a-z]* \d{2,4}\b'
               r'|\b(?:jan|feb|mar|apr|may|jun|jul|aug|sep|oct|nov|dec)[a-z]* \d{1,2},? \d{4}\b',
    time     = r'\b\d{1,2}:\d{2}(?::\d{2})?\s?(?:am|pm)?\b',
    ref      = r'\b(?:invoice|inv|order|p\.?o|ref(?:erence)?|receipt|bill|account|sku|part|item)\s*'
               r'(?:no\.?|number|id)?\s*[:#]\s*[a-z0-9][a-z0-9\-/]{2,}\b',
    qty      = r'\b(?:qty|quantity|units?|pcs?|nos?)\b\s*[:.]?\s*\d+|\b\d+\s?(?:x|×)\s?\d',
    percent  = r'\b\d{1,3}(?:\.\d+)?\s?%',
    tax      = r'\b(?:vat|gst|hsn|sac|tin|ein|abn|sales tax|tax id|withholding)\b',
    email    = r'\b[\w.+-]+@[\w-]+\.[\w.]{2,}\b',
    url      = r'https?://\S+',
    citation = r'\[\d+\]|\bet al\.|\bdoi:|\barxiv:',
    code     = r'^\s*(?:def|class|function|const|let|var|import|from|package|#include)\s+\w',
    table    = r'^\s*\|.+\|\s*$',
    heading  = r'^#{1,6} \S',
    blank    = r'_{4,}|\[ ?\]',
)
_SIG = {k: re.compile(v, re.I|re.M) for k, v in SIGNALS.items()}
# entity extraction runs on the head of a document (`NER_CHARS`, from rahasya): that is where
# its type is decided

# handrolled entity extraction: ORG by legal suffix, PERSON by honorific, LAW by instrument type
_ORG_SUFF = re.compile(
    r'(?<![a-z,\.])([A-Z][A-Za-z0-9\'\-& ]{1,50}?)\s+'
    r'(?:Ltd\.?|Limited|Corp\.?|Corporation|Inc\.?|LLC|LLP|GmbH|AG|SA|NV|BV|Pty\.?|PLC'
    r'|Group\b|Holdings?\b|Associates?\b|Partners?\b|Ventures?\b'
    r'|University\b|College\b|Institute\b|Hospital\b|Foundation\b'
    r'|Bank\b|Fund\b|Trust\b|Council\b|Authority\b|Commission\b'
    r'|Ministry\b|Department\b|Agency\b|Bureau\b)\b'
)
# the PERSON arm is rahasya's, so a name means the same thing to the doctype scorer and to the
# privacy gate. One regex, measured once.
_PERSON_HON = PERSON_HON
_LAW_INST = re.compile(
    r'\b([A-Z][A-Za-z ]{2,50}?)\s+'
    r'(?:Act(?:\s+\d{4})?|Regulations?\b|Directive\b|Ordinance\b|Statute\b|Treaty\b|Convention\b|Amendment\b)'
)

def _noun_ents(text:str, limit:int=40) -> L:
    'ORG, PERSON and LAW entities via regex; no model weight.'
    seen, out = set(), L()
    def _add(t, label):
        n = re.sub(r'\s+', ' ', t.strip())[:60]
        k = n.lower()
        if n and k not in seen: seen.add(k); out.append((n, label))
    for m in _ORG_SUFF.finditer(text): _add(m.group(0).strip(), 'ORG')
    for m in _PERSON_HON.finditer(text): _add(m.group(1), 'PERSON')
    for m in _LAW_INST.finditer(text): _add(m.group(1).strip(), 'LAW')
    return out[:limit]

def _keyphrases(text:str) -> tuple:
    'litesearch keyphrases when it is installed, nothing when it is not. Display only: `cue_scores` reads the ORG/PERSON/LAW labels, never these.'
    try: from litesearch import text_entities
    except ImportError: return ()
    return text_entities(text)

def signals(text:str,            # the document text
            ner:bool=True,       # run entity extraction (noun entities + keyphrases)
            limit:int=20,        # entities kept
) -> AttrDict:
    'Countable evidence in one document: `counts` per signal, plus the entities it names.'
    counts = {k: len(p.findall(text or '')) for k, p in _SIG.items()}
    noun = L(_noun_ents((text or '')[:NER_CHARS]) if ner else ()).map(
        lambda t: AttrDict(label=t[1], text=t[0]))
    kws  = L(_keyphrases((text or '')[:NER_CHARS]) if ner else ()).map(
        lambda t: AttrDict(label=(t[1] or 'KEYPHRASE').upper(), text=re.sub(r'\s+', ' ', t[0].strip())[:60])
    ).filter(lambda e: e.text and e.label not in {'ORG', 'PERSON', 'LAW', 'PRODUCT', 'GPE'})
    found = noun + kws
    labels = dict(Counter(e.label.lower() for e in found))
    # labels count everything found; limit caps display only (cue_scores reads labels)
    if noun:
        counts = {k: counts.get(k, 0) + (labels.get(k, 0) if k not in counts else 0)
                  for k in (*counts, *(l for l in labels if l not in ('keyphrase', 'term')))}
    return AttrDict(counts={k: v for k, v in counts.items() if v}, labels=labels,
                    ents=found.sorted(key=lambda e: e.label in ('KEYPHRASE', 'TERM'))[:limit],
                    method=('ner+regex' if noun else 'keyphrase+regex' if kws else 'regex'))

## Doctypes: what a document is

In [ ]:
#| export
DOCTYPES = {
    # label: the cue phrases that are evidence for it, the regex signals it needs, and the
    # entity labels it expects
    'invoice': dict(needs=('money',), ents=('ORG',), cues=(
        r'\b(?:tax )?invoice\b', r'\b(?:amount|balance|total) due\b', r'\bbill(?:ed)? to\b|\bremit\b',
        r'\bpayment terms?\b|\bnet \d{1,3}\b', r'\bsub-?total\b', r'\b(?:vat|gst|sales tax)\b')),
    'receipt': dict(needs=('money',), ents=('ORG',), cues=(
        r'\breceipt\b', r'\bthank you for your (?:order|purchase|payment)\b',
        r'\bcard\b[^\n]{0,16}\bending\b|\bcash tendered\b|\bchange due\b',
        r'\btransaction (?:id|no)\b|\bauth(?:orisation|orization) code\b',
        r'\bpaid\b|\bpayment received\b', r'\bmerchant\b|\bstore #\s?\d+\b')),
    'purchase_order': dict(needs=('ref',), ents=('ORG',), cues=(
        r'\bpurchase order\b|\bp\.?o\.?\s*(?:no|number|#)', r'\bship(?:[ -]?to)?\b',
        r'\bdelivery date\b|\brequested delivery\b', r'\bvendor\b|\bsupplier\b',
        r'\brequisition\b', r'\bunit price\b')),
    'quote': dict(needs=('money',), ents=('ORG',), cues=(
        r'\bquotation\b|\bquote\s*(?:no|number|#)|\bestimate\b', r'\bvalid (?:until|for|through)\b',
        r'\bunit price\b', r'\bterms and conditions\b',
        r'\bwe are pleased to (?:quote|offer)\b|\bno obligation\b')),
    'catalogue': dict(needs=('money',), ents=(), cues=(
        r'\bcatalog(?:ue)?\b|\bprice list\b|\bproduct list\b',
        r'\bsku\b|\bmodel\s*(?:no|number)\b|\bpart\s*(?:no|number)\b',
        r'\b(?:in|out of) stock\b|\bavailability\b', r'\bper (?:unit|pack|case|kg|litre|liter)\b',
        r'\bspecifications?\b|\bdimensions\b', r'\badd to (?:cart|basket)\b')),
    'contract': dict(needs=(), ents=('ORG', 'LAW'), cues=(
        r'\bagreement\b|\bcontract\b', r'\bparties\b|\bby and between\b',
        r'\bhereby\b|\bwhereas\b|\bhereinafter\b', r'\bshall\b',
        r'\bgoverning law\b|\bjurisdiction\b|\btermination\b',
        r'\bconfidential(?:ity)?\b|\bindemnif|\bliability\b')),
    'resume': dict(needs=('email',), ents=('PERSON', 'ORG'), cues=(
        r'\b(?:curriculum vitae|resum[eé])\b',
        r'\bwork experience\b|\bemployment history\b|\bprofessional experience\b',
        r'\beducation\b', r'\bskills\b', r'\bcertification',
        r'\breferences available\b|linkedin\.com/in/')),
    'paper': dict(needs=('citation',), ents=('ORG', 'PERSON'), cues=(
        r'\babstract\b', r'\bintroduction\b', r'\brelated work\b|\bmethodology\b|\bexperiments?\b',
        r'\breferences\b|\bbibliography\b', r'\bwe (?:propose|present|show|evaluate)\b',
        r'\bdoi:|\barxiv:')),
    'report': dict(needs=(), ents=('ORG',), cues=(
        r'\bexecutive summary\b', r'\bfindings\b|\bconclusions?\b|\brecommendations?\b',
        r'\bq[1-4] \d{4}\b|\bfiscal year\b|\bfy\d{2}\b|\bquarter\b',
        r'\btable \d+\b|\bfigure \d+\b|\bappendix\b',
        r'\byear[- ]on[- ]year\b|\brevenue\b|\bmargin\b')),
    # --- work-product labels ---
    'proposal': dict(needs=(), ents=('ORG',), cues=(
        r'\bproposal\b|\brfp response\b|\brequest for proposal\b', r'\bproposed approach\b|\bour approach\b',
        r'\bscope of work\b|\bwork ?plan\b', r'\bdeliverables?\b', r'\bfees?\b|\binvestment\b',
        r'\bproject team\b|\bteam structure\b')),
    'presentation': dict(needs=(), ents=(), cues=(
        r'\bslide deck\b|\bthis (?:deck|presentation)\b|\bslide \d+\b',
        r'\bkey takeaways?\b|\bkey messages?\b', r'\bfor discussion\b|\bquestions? for discussion\b',
        r'\bexecutive update\b|\bsteering committee\b|\bboard update\b',
        r'\bappendix slides?\b|\bback-?up slides?\b',
        r'\bagenda for (?:today|this session)\b|\bwalk(?:ing)? through\b')),
    'requirements_spec': dict(needs=(), ents=(), cues=(
        r'\brequirements? specification\b|\bproduct requirements?\b', r'\bacceptance criteria\b',
        r'\bfunctional requirements?\b|\bnon[- ]functional requirements?\b', r'\buser stor(?:y|ies)\b',
        r'\brequirement id\b|\btraceability matrix\b', r'\bthe system shall\b|\bmust be able to\b')),
    'technical_design': dict(needs=(), ents=(), cues=(
        r'\bsolution design\b|\btechnical design\b|\bdesign document\b',
        r'\bdata architecture\b|\bsystem architecture\b|\barchitecture (?:diagram|overview)\b',
        r'\bdata flow\b|\bsequence diagram\b|\bcomponent (?:diagram|view)\b',
        r'\bapplication hosting\b|\bnon[- ]functional\b|\bscalability\b',
        r'\bintegration (?:points?|patterns?|layer)\b|\binterface (?:contract|spec)\b',
        r'\bsubnet\b|\bapi gateway\b|\bdeployment (?:topology|diagram)\b')),
    'regulatory_guidance': dict(needs=(), ents=('LAW', 'ORG'), cues=(
        r'\bregulation \(?(?:eu|ec)\b|\bmedical device regulation\b', r'\bcompetent authority\b',
        r'\bnotified bod(?:y|ies)\b|\bconformity assessment\b', r'\bauthori[sz]ed representative\b',
        r'\btechnical documentation\b|\btechnical file\b', r'\barticle \d+\b|\bannex [ivx]+\b')),
    'procedure': dict(needs=(), ents=('ORG',), cues=(
        r'\bstandard operating procedure\b|\bs\.?o\.?p\.?\b|\bwork instruction\b',
        r'\bprocess owner\b|\bresponsibilit(?:y|ies)\b', r'\bapproval workflow\b|\bapproval authority\b',
        r'\bescalation\b|\bexception handling\b', r'\brevision history\b|\bchange control\b',
        r'\bthis procedure (?:applies|describes|covers)\b|\bscope and purpose\b')),
    'qa_artifact': dict(needs=(), ents=(), cues=(
        r'\btest (?:case|plan|script|protocol)\b', r'\bexpected results?\b|\bactual results?\b',
        r'\bsteps to reproduce\b|\bdefects?\b|\bbug report\b',
        r'\bdeviation log\b|\bdiscrepanc(?:y|ies)\b',
        r'\btemplate fidelity\b|\bvalidation report\b|\bqa audit\b',
        r'\bpass/fail\b|\btest (?:results?|status|evidence)\b')),
    'roadmap': dict(needs=(), ents=('ORG',), cues=(
        r'\broadmap\b', r'\bmilestones?\b', r'\bphase [0-9a-z]+\b|\bnow[,/ ]+next[,/ ]+later\b',
        r'\bnext (?:30|60|90) days\b|\bquarterly plan\b', r'\btarget date\b|\bplanned date\b',
        r'\bproduct backlog\b|\bdelivery plan\b|\bthemes?\b')),
    # claim vs clinical_record share a dated-person shape; cues must separate them
    'claim': dict(needs=('date',), ents=('PERSON', 'ORG'), cues=(
        r'\bclaim (?:number|no\.?|form|reference)\b|\bclaim id\b',
        r'\bincident date\b|\bdate of loss\b', r'\bclaimant\b|\binsured\b|\bpolicyholder\b',
        r'\bpolicy (?:number|no\.?|#)\b|\bcoverage\b', r'\b(?:loss )?adjuster\b|\bassessor\b',
        r'\bsettlement\b|\bdeductible\b|\bexcess\b|\bliability accepted\b')),
    'clinical_record': dict(needs=('date',), ents=('PERSON', 'ORG'), cues=(
        r'\bpatient\b|\bencounter\b', r'\bdiagnos(?:is|es|ed)\b|\bicd-?10\b',
        r'\bpresenting complaint\b|\bclinical (?:history|notes?)\b',
        r'\bprescri(?:bed|ption)\b|\bmedication\b|\bdosage\b',
        r'\bdischarge summary\b|\breferral letter\b|\bclinician\b|\bphysician\b',
        r'\badverse events?\b|\bdevice malfunction\b|\bside effects?\b')),
    'documentation': dict(needs=('heading',), ents=(), cues=(
        r'\binstallation\b|\bgetting started\b|\bquick ?start\b',
        r'\busage\b|\bexamples?\b|\bapi reference\b', r'\bparameters?\b|\breturns\b|\barguments?\b',
        r'\bpip install\b|\bnpm install\b|\bdocker run\b', r'\bconfiguration\b|\bsee also\b',
        r'^```')),
    'code': dict(needs=('code',), ents=(), cues=(
        r'^\s*(?:def|class)\s+\w+|^\s*(?:function|const|let|var)\s+\w+',
        r'^\s*(?:import|from|#include|package|use)\s+\w', r'\breturn\b',
        r'^\s*(?://|#|/\*)', r'\b(?:if|for|while)\s*\(', r'\b(?:public|private|static|async)\b')),
    'email': dict(needs=('email',), ents=('PERSON',), cues=(
        r'^(?:from|to|cc|bcc|subject|sent):', r'^\s*(?:dear|hi|hello)\b',
        r'\b(?:best|kind) regards\b|\bsincerely\b|\bthanks,\b',
        r'\bforwarded message\b|\bwrote:\s*$', r'\bunsubscribe\b')),
    'meeting_notes': dict(needs=(), ents=('PERSON',), cues=(
        r'\b(?:meeting|call) notes\b|\bminutes\b', r'\battendees\b|\bparticipants\b|\bpresent\b',
        r'\bagenda\b', r'\baction items?\b|\bnext steps\b|\bfollow[- ]ups?\b',
        r'\bdecided\b|\bdecisions?\b|\bowner\b|\bdue by\b')),
    'form': dict(needs=('blank',), ents=(), cues=(
        r'\bplease (?:complete|fill|print|sign)\b', r'\bfor office use only\b',
        r'\bapplicant\b|\bapplication (?:form|for)\b', r'\bsignature\b|\bdate signed\b',
        r'\b(?:full name|surname|given names?|date of birth|dob)\b', r'_{4,}')),
    'transcript': dict(needs=('time',), ents=('PERSON',), cues=(
        r'\btranscript\b', r'^\s*\[?\d{1,2}:\d{2}', r'^\s*(?:speaker \d|[A-Z][a-z]+\s?[A-Z]?[a-z]*):',
        r'\b(?:um|uh|you know|i mean)\b', r'\bwelcome (?:back|to)\b|\bthanks for (?:watching|listening)\b')),
    'article': dict(needs=(), ents=('PERSON', 'ORG'), cues=(
        r'\bpublished\b|\bposted (?:on|by)\b', r'\bread more\b|\bshare this\b|\bcomments?\b',
        r'\bsubscribe\b|\bnewsletter\b', r'\baccording to\b', r'\btags?:|\bcategor(?:y|ies):')),
}
_CUES = {l: [re.compile(c, re.I|re.M) for c in d['cues']] for l, d in DOCTYPES.items()}

# arrival kind is strong evidence (youtube -> transcript)
KIND_HINT = dict(youtube='transcript', arxiv='paper', code='code')
MIN_SCORE, MIN_MARGIN, KIND_BONUS = 0.4, 0.12, 0.2

def cue_scores(text:str, sig=None) -> dict:
    'Every doctype scored against `text`, best first: the categorisation no model is needed for.'
    sig = sig if sig is not None else signals(text)
    ner, out = sig.method.startswith('ner'), {}
    for lbl, d in DOCTYPES.items():
        cues = [p for p in _CUES[lbl] if p.search(text or '')]
        need = [s for s in d['needs'] if sig.counts.get(s)]
        ents = [e for e in d['ents'] if sig.labels.get(e.lower())] if ner else []
        legs = [(0.6, len(cues)/len(_CUES[lbl]))]
        if d['needs']: legs.append((0.25, len(need)/len(d['needs'])))
        if ner and d['ents']: legs.append((0.15, len(ents)/len(d['ents'])))
        w = sum(x for x, _ in legs)
        out[lbl] = round(sum(x*f for x, f in legs)/w, 3)
    return dict(sorted(out.items(), key=lambda kv: -kv[1]))

def guess_type(text:str,             # the document text
               sig=None,             # a signals() result, computed if not given
               kind:str=None,        # the vault kind that acquired it, if known
               min_score:float=MIN_SCORE,    # below this the cues have not decided
               min_margin:float=MIN_MARGIN,  # runner-up this close means they have not either
) -> AttrDict:
    'The best doctype for `text` from cues alone, and whether that guess is worth trusting.'
    sig = sig if sig is not None else signals(text)
    sc = cue_scores(text, sig)
    if kind in KIND_HINT: sc[KIND_HINT[kind]] = round(min(1.0, sc[KIND_HINT[kind]] + KIND_BONUS), 3)
    ranked = sorted(sc.items(), key=lambda kv: -kv[1])
    (top, best), (_, second) = ranked[0], (ranked[1] if len(ranked) > 1 else ('', 0.0))
    return AttrDict(doctype=top if best else 'other', score=best, margin=round(best-second, 3),
                    decisive=best >= min_score and best-second >= min_margin,
                    scores=dict(ranked[:5]), method=sig.method)

## Tests

In [ ]:
from fastcore.test import test_eq, test_fail

### The work-product labels

In [ ]:
#| hide
# The cue table is a pure function of the text, so unlike the `signals` tests further down
# this one can run without the NER model: `ner=False` drops the entity leg and leaves cues and
_gt = lambda t: guess_type(t, sig=signals(t, ner=False))

WORK = {
 'proposal': "# Proposal\nRequest for proposal response.\nOur proposed approach and scope of work.\n"
     "Work plan and deliverables.\nFees: $120,000 investment.\nProject team and team structure.\n",
 'presentation': "# Slide deck\nExecutive update to the steering committee.\nKey takeaways and key "
     "messages.\nQuestions for discussion.\nAppendix slides.\nWalking through slide 4.\n",
 'requirements_spec': "# Requirements Specification\nProduct requirements.\nFunctional requirements "
     "and non-functional requirements.\nUser stories and acceptance criteria.\nRequirement ID "
     "R-101. Traceability matrix.\nThe system shall log every access.\n",
 'technical_design': "# Solution Design\nTechnical design document.\nSystem architecture and data "
     "architecture.\nData flow and sequence diagram; component diagram.\nApplication hosting and "
     "scalability.\nIntegration points and interface contract.\nSubnet, API gateway, deployment "
     "topology.\n",
 'regulatory_guidance': "# Guidance\nRegulation (EU) 2017/745, the medical device regulation.\n"
     "The competent authority and the notified body.\nConformity assessment and the authorised "
     "representative.\nTechnical documentation, the technical file.\nArticle 10 and Annex IX.\n",
 'procedure': "# Standard Operating Procedure\nScope and purpose: this procedure describes the "
     "process.\nProcess owner and responsibilities.\nApproval workflow and approval authority.\n"
     "Escalation and exception handling.\nRevision history and change control.\n",
 'qa_artifact': "# Test Plan\nTest case TC-01.\nExpected result and actual result.\nSteps to "
     "reproduce; defects raised as a bug report.\nDeviation log and discrepancies.\nTemplate "
     "fidelity checked; validation report and QA audit.\nPass/fail test status recorded.\n",
 'roadmap': "# Roadmap\nMilestones by phase 1 and phase 2.\nNext 90 days; quarterly plan.\nTarget "
     "date and planned date per item.\nProduct backlog and delivery plan by theme.\n",
 'claim': "# Claim\nClaim number CLM-88. Claim form attached.\nIncident date 2024-02-01, date of "
     "loss 2024-02-01.\nClaimant and insured; policyholder notified. Policy number POL-42.\n"
     "Loss adjuster assigned.\nSettlement after the deductible.\n",
 'clinical_record': "# Discharge summary\nPatient encounter on 2024-02-01.\nPresenting complaint "
     "and clinical history.\nDiagnosis recorded; ICD-10 coded.\nMedication and prescription, with "
     "dosage.\nClinician and physician sign-off.\nAdverse event and device malfunction reported.\n",
}
# each of the ten wins its own kind, and wins it decisively enough to need no model
for want, txt in WORK.items():
    r = _gt(txt)
    test_eq((want, r.doctype), (want, want))
    assert r.decisive, (want, r.scores)

# ...and prose that merely borrows the vocabulary is none of them
NEUTRAL = ("The deploy pipeline runs on GitHub Actions. Each component is integrated and the test "
           "plan passed on the first run; one job failed and was retried. See the appendix for "
           "next steps and the list of dependencies. This procedure is documented, with an "
           "assessment of the estimate.\n")
_sc = cue_scores(NEUTRAL, signals(NEUTRAL, ner=False))
assert not _gt(NEUTRAL).decisive, _sc
for _l in WORK: assert _sc[_l] < 0.2, (_l, _sc[_l])   # every one of the ten, well under MIN_SCORE

# the incumbents the new labels are closest to must still win their own documents
test_eq(_gt("# Quotation\nQuote number Q-77.\nValid until 2024-06-01.\nUnit price $8.50.\nTerms "
            "and conditions apply.\nWe are pleased to quote, no obligation. An estimate follows.\n"
            ).doctype, 'quote')
_mn = _gt("# Meeting notes\nAttendees: Ada, Bob\nAgenda\n- the roadmap\nAction items:\n"
          "- Bob to send the deck\nNext steps: review the milestones and dependencies, due by "
          "Friday. Decided: ship.\n")
test_eq(_mn.doctype, 'meeting_notes')          # `roadmap` shares half its vocabulary...
assert _mn.margin >= MIN_MARGIN, _mn.scores    # ...and must not take the rest of its headroom

# claim + clinical_record tie: not decisive, costs a model call
_both = _gt("# Claim\nClaim number CLM-88. Claim form attached.\nIncident date 2024-02-01, date of "
            "loss 2024-02-01.\nClaimant and insured details.\nPatient encounter, diagnosis noted. "
            "Adverse event and device malfunction.\n")
test_eq(sorted(list(_both.scores)[:2]), ['claim', 'clinical_record'])
test_eq((_both.margin, _both.decisive), (0.0, False))

In [ ]:
#| hide
# the invoice family, and the neighbours each one has to beat
_g = lambda t: guess_type(t, sig=signals(t, ner=False))
test_eq(_g('# INVOICE\nInvoice No: ACM-2024-0117\nTotal due: 1,240.00 EUR\n'
           'Payment terms: net 30\nBill to: Contoso GmbH\n').doctype, 'invoice')
test_eq(_g('# Receipt\nThank you for your purchase.\nPaid by card ending 1111.\n'
           'Change due 0.00. Subtotal 12.00, VAT 2.40.\n').doctype, 'receipt')
test_eq(_g('# Abstract\nWe present a method. Related work [1], see et al.\n'
           'doi: 10.1000/x. Our results and conclusion follow.\n').doctype, 'paper')

In [ ]:
#| hide
# entities are counted into the signal table, and reach the scorer as labels
_s = signals('Dr Charles Babbage of Acme Supplies Ltd cited the Data Protection Act 2018.')
# keyphrases come and go with litesearch; the three the scorer reads do not
test_eq({'law', 'org', 'person'} <= set(_s.labels), True)
test_eq({'ORG', 'PERSON', 'LAW'} <= {e.label for e in _s.ents}, True)
# `ner=False` drops the entity leg entirely, and says so
test_eq(signals('Dr Charles Babbage of Acme Supplies Ltd.', ner=False).method, 'regex')
test_eq(signals('Dr Charles Babbage of Acme Supplies Ltd.', ner=False).labels, {})

In [ ]:
#| hide
# the name regex is rahasya's, so the two packages cannot drift apart on what a name is
import rahasya
test_eq(_PERSON_HON is rahasya.PERSON_HON, True)
test_eq(signals('Dr Charles Babbage signed it.').ents[0].text, 'Charles Babbage')

In [ ]:
#| hide
# keyphrases are optional: with litesearch absent, `signals` still scores every doctype
import builtins
_real = builtins.__import__
def _no_litesearch(name, *a, **kw):
    if name == 'litesearch': raise ImportError('litesearch')
    return _real(name, *a, **kw)
builtins.__import__ = _no_litesearch
try:
    test_eq(_keyphrases('an invoice for widgets'), ())
    test_eq(guess_type(WORK['proposal'], sig=signals(WORK['proposal'])).doctype, 'proposal')
finally: builtins.__import__ = _real

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()